In [ ]:
rdm_url = 'http://localhost:5000/'
idp_name_1 = 'FakeCAS'
idp_username_1 = None
idp_password_1 = None
rdm_project_name = 'TEST-Migration-S3'
target_storage_name = 'Amazon S3'
target_file_name = '移行前 + データ.txt'
target_file_content = 'S3 storage migration fixture'
default_result_path = None
close_on_fail = False
transition_timeout = 60000
s3_access_key = None
s3_secret_key = None
s3_bucket = None

In [ ]:
from getpass import getpass
from pathlib import Path
from tempfile import mkdtemp
import sys

if idp_username_1 is None:
    idp_username_1 = input('Username: ')
if idp_password_1 is None:
    idp_password_1 = getpass(f'Password for {idp_username_1}: ')

sys.path.insert(0, str(Path.cwd().parent))
from playwright.async_api import expect
from scripts import grdm
from scripts.playwright import init_pw_context, run_pw, finish_pw_context

work_dir = Path(mkdtemp())
await init_pw_context(close_on_fail=close_on_fail, last_path=default_result_path or str(work_dir))
work_dir

if s3_access_key is None:
    s3_access_key = input('S3 access key: ')
if s3_secret_key is None:
    s3_secret_key = getpass('S3 secret key: ')
if s3_bucket is None:
    s3_bucket = input('S3 bucket: ')

# 旧版でAmazon S3に接続してファイルを登録する

- サブシステム名: ストレージ
- ページ/アドオン: Amazon S3
- 機能分類: 移行互換性
- 用意するテストデータ: 移行試験用GRDM、ユーザー、S3バケット
- シナリオ名: 旧版でのS3接続とファイルの登録

## GRDMトップページを開く

ログイン画面へ移動できること。

In [ ]:
async def _step(page):
    await page.goto(rdm_url)
    consent = page.get_by_role('button', name='同意する', exact=True)
    sign_in = page.locator('[data-test-sign-in-button]')
    await expect(page.locator('button:visible').and_(consent).or_(sign_in).first).to_be_visible(
        timeout=transition_timeout)
    if await consent.is_visible():
        await consent.click()
    await expect(page.locator('[data-test-sign-in-button]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## GRDMにログインする

マイプロジェクトが表示されること。

In [ ]:
async def _step(page):
    await grdm.login(page, idp_name_1, idp_username_1, idp_password_1, transition_timeout=transition_timeout)
    await expect(page.locator('[data-test-create-project-modal-button]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 移行試験用のプロジェクトを作る

マイプロジェクトに作成したプロジェクトが表示されること。

In [ ]:
async def _step(page):
    await grdm.ensure_project_exists(page, rdm_project_name, transition_timeout=transition_timeout)
    await expect(page.locator('[data-test-dashboard-item-title]').filter(has_text=rdm_project_name)).to_be_visible()

await run_pw(_step)

## マイプロジェクトから対象のプロジェクトを開く

プロジェクトのファイルページを開けること。

In [ ]:
async def _step(page):
    await page.locator('[data-test-dashboard-item-title]').filter(has_text=rdm_project_name).click()
    await expect(page.locator('#projectNavFiles a')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## プロジェクトのアドオン画面でAmazon S3を有効にする

アカウントに接続するためのリンクが表示されること。

In [ ]:
async def _step(page):
    await grdm.enable_addon(page, target_storage_name, transition_timeout=transition_timeout)
    await expect(page.locator('#s3Scope').get_by_text('アカウントに接続する', exact=False)).to_be_visible(
        timeout=transition_timeout)

await run_pw(_step)

## 「アカウントに接続する」をクリックする

アクセスキーとシークレットキーの入力欄が表示されること。

In [ ]:
async def _step(page):
    await page.locator('#s3Scope').get_by_text('アカウントに接続する', exact=False).click()
    await expect(page.locator('#s3InputCredentials #access_key')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## アクセスキーとシークレットキーを入力して「保存」をクリックする

アカウント設定が保存され、「接続」をクリックできること。

In [ ]:
async def _step(page):
    credentials = page.locator('#s3InputCredentials')
    await credentials.locator('#access_key').fill(s3_access_key)
    await credentials.locator('#secret_key').fill(s3_secret_key)
    await credentials.locator('.btn-success').click()
    await expect(page.get_by_role('button', name='接続', exact=True)).to_be_enabled(timeout=transition_timeout)

await run_pw(_step)

## 「接続」をクリックして試験用バケットを選択する

試験用バケットを選択できること。

In [ ]:
async def _step(page):
    await page.get_by_role('button', name='接続', exact=True).click()
    bucket = page.locator(f'//span[text() = "{s3_bucket}"]/../../..//input[@type = "radio"]')
    await bucket.check()
    await expect(bucket).to_be_checked()

await run_pw(_step)

## バケットの設定を保存する

バケットがプロジェクトにリンクされたことが表示されること。

In [ ]:
async def _step(page):
    await page.locator('.s3-confirm-selection input[value="保存"]').click()
    await expect(page.locator('#s3Scope > .help-block').get_by_text('正常にリンクされました')).to_be_visible(
        timeout=transition_timeout)

await run_pw(_step)

## 上部メニューから「ファイル」をクリックする

既存のAmazon S3接続先が表示されること。

In [ ]:
async def _step(page):
    await page.locator('#projectNavFiles a').click()
    await expect(grdm.get_select_expanded_storage_title_locator(page, target_storage_name)).to_be_visible(
        timeout=transition_timeout)

await run_pw(_step)

## Amazon S3へ移行試験用のファイルをアップロードする

ファイルがアップロードされ、一覧に表示されること。

In [ ]:
async def _step(page):
    path = work_dir / target_file_name
    path.write_text(target_file_content, encoding='utf-8')
    await grdm.get_select_storage_title_locator(page, target_storage_name).click()
    await grdm.upload_file(page, str(path))
    await grdm.wait_for_uploaded(page, target_file_name, timeout=transition_timeout)

await run_pw(_step)

In [ ]:
await finish_pw_context()